# Budget-Based Vehicle Recommender

Train this notebook on Kaggle, then download the saved model and data to deploy on Streamlit Cloud.

What the model does:
- Takes your budget + preferences
- Recommends vehicles you can afford, ranked by **advantage score**
- Outputs `model/vehicle_model.joblib` and `model/vehicle_data.csv`

In [ ]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import NearestNeighbors
import joblib
import os


In [ ]:
# Upload data/vehicles.csv to this Kaggle notebook first
df = pd.read_csv("vehicles.csv")
print(df.shape)
print(df.head())


In [ ]:
feature_cols = ["Price_USD", "Mileage_kmpl", "Seats", "Advantage_Score"]
categorical_cols = ["Type", "Fuel_Type"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", MinMaxScaler(), feature_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
    ],
    remainder="drop",
)

nn = NearestNeighbors(n_neighbors=min(5, len(df)), metric="euclidean")
pipe = Pipeline(steps=[("prep", preprocessor), ("nn", nn)])
pipe.fit(df[feature_cols + categorical_cols])

os.makedirs("/kaggle/working/model", exist_ok=True)
joblib.dump(
    {
        "pipe": pipe,
        "feature_cols": feature_cols,
        "categorical_cols": categorical_cols,
    },
    "/kaggle/working/model/vehicle_model.joblib",
)
df.to_csv("/kaggle/working/model/vehicle_data.csv", index=False)
print("Saved to /kaggle/working/model/")


In [ ]:
!ls -la /kaggle/working/model/